In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 100

data = pd.DataFrame({
    "student_id": range(1, n+1),
    "age": np.random.randint(21, 35, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "country_applied": np.random.choice(["USA", "Canada", "UK", "Australia", "Germany"], n),
    "course": np.random.choice(["Data Science", "MBA", "Engineering", "IT", "Finance"], n),
    "university_rank": np.random.randint(1, 500, n),
    "ielts_score": np.round(np.random.uniform(5.5, 8.5, n), 1),
    "work_experience": np.random.randint(0, 5, n),
    "previous_degree": np.random.choice(["B.Tech", "B.Com", "BBA", "BA"], n),
    "loan_amount": np.random.randint(500000, 3000000, n),
    "placement_support": np.random.choice([0, 1], n),
    "internship_experience": np.random.choice([0, 1], n),
    "cgpa": np.round(np.random.uniform(6.0, 9.5, n), 2)
})

# Target variable logic (job_offer)
data["job_offer"] = (
    (data["ielts_score"] > 6.5) &
    (data["cgpa"] > 7.0) &
    (data["placement_support"] == 1)
).astype(int)

# Regression target (expected salary)
data["expected_salary"] = (
    30000 +
    (data["ielts_score"] * 5000) +
    (data["cgpa"] * 4000) +
    (data["work_experience"] * 6000) +
    (data["job_offer"] * 15000)
).astype(int)

print(data.head())

   student_id  age  gender country_applied        course  university_rank  \
0           1   27    Male              UK           MBA              355   
1           2   24  Female         Germany  Data Science              306   
2           3   33    Male       Australia            IT              409   
3           4   31  Female         Germany            IT              408   
4           5   28  Female              UK  Data Science               13   

   ielts_score  work_experience previous_degree  loan_amount  \
0          8.3                4              BA      1008153   
1          7.4                1          B.Tech      1860642   
2          7.1                1          B.Tech      1187409   
3          7.5                1              BA      1140053   
4          6.8                4              BA      1360395   

   placement_support  internship_experience  cgpa  job_offer  expected_salary  
0                  0                      1  6.65          0           1

In [2]:
data.head()

,student_id,age,gender,country_applied,course,university_rank,ielts_score,work_experience,previous_degree,loan_amount,placement_support,internship_experience,cgpa,job_offer,expected_salary
0,1,27,Male,UK,MBA,355,8.3,4,BA,1008153,0,1,6.65,0,122100
1,2,24,Female,Germany,Data Science,306,7.4,1,B.Tech,1860642,0,0,6.82,0,100280
2,3,33,Male,Australia,IT,409,7.1,1,B.Tech,1187409,0,1,8.22,0,104380
3,4,31,Female,Germany,IT,408,7.5,1,BA,1140053,1,1,9.18,1,125220
4,5,28,Female,UK,Data Science,13,6.8,4,BA,1360395,0,1,7.11,0,116440


In [3]:
data.isnull().sum()

student_id               0
age                      0
gender                   0
country_applied          0
course                   0
university_rank          0
ielts_score              0
work_experience          0
previous_degree          0
loan_amount              0
placement_support        0
internship_experience    0
cgpa                     0
job_offer                0
expected_salary          0
dtype: int64

In [4]:
data.duplicated().sum()

0

In [5]:
data.describe()

,student_id,age,university_rank,ielts_score,work_experience,loan_amount,placement_support,internship_experience,cgpa,job_offer,expected_salary
count,100.000000,100.000000,100.000000,100.00000,100.000000,1.000000e+02,100.000000,100.000000,100.00000,100.000000,100.000000
mean,50.500000,27.880000,241.250000,7.00800,2.160000,1.825873e+06,0.490000,0.520000,7.76190,0.190000,111897.600000
std,29.011492,4.040902,143.359221,0.85383,1.440539,7.091453e+05,0.502418,0.502117,0.92757,0.394277,12336.808775
min,1.000000,21.000000,1.000000,5.50000,0.000000,5.069490e+05,0.000000,0.000000,6.04000,0.000000,83960.000000
25%,25.750000,24.000000,119.500000,6.30000,1.000000,1.195478e+06,0.000000,0.000000,6.98250,0.000000,103420.000000
50%,50.500000,28.000000,263.000000,7.10000,2.000000,1.801155e+06,0.000000,1.000000,7.87000,0.000000,112770.000000
75%,75.250000,32.000000,354.250000,7.62500,3.000000,2.467588e+06,1.000000,1.000000,8.51750,0.000000,120460.000000
max,100.000000,34.000000,494.000000,8.40000,4.000000,2.962492e+06,1.000000,1.000000,9.49000,1.000000,141880.000000


In [6]:
data["country_applied"].value_counts()

country_applied
USA          26
Australia    25
UK           23
Canada       15
Germany      11
Name: count, dtype: int64

In [7]:
import pandas as pd

def preprocess_data(df, target="expected_salary"):
    import pandas as pd
    """
    This function:
    1. Encodes categorical variables
    2. Splits data into X (features) and y (target)

    Parameters:
    ----------
    df : pandas DataFrame
    target : target column name (default = expected_salary)

    Returns:
    -------
    X : Features (encoded)
    y : Target variable
    """

    df = df.copy()  # avoid modifying original data

    # Drop ID column (not useful for ML)
    if "student_id" in df.columns:
        df.drop("student_id", axis=1, inplace=True)

    # Separate categorical columns
    categorical_cols = ["gender", "country_applied", "course", "previous_degree"]

    # Apply One-Hot Encoding
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    """
    Why drop_first=True?
    ---------------------
    Avoids dummy variable trap (multicollinearity)
    Important for linear models
    """

    # Split into X and y
    X = df_encoded.drop(target, axis=1)
    y = df_encoded[target]

    return X, y

In [8]:
X, y = preprocess_data(data, target="expected_salary")

print(X.shape)
print(y.shape)

(100, 21)
(100,)


In [9]:
X

,age,university_rank,ielts_score,work_experience,loan_amount,placement_support,internship_experience,cgpa,job_offer,gender_Male,...,country_applied_Germany,country_applied_UK,country_applied_USA,course_Engineering,course_Finance,course_IT,course_MBA,previous_degree_B.Tech,previous_degree_BA,previous_degree_BBA
0,27,355,8.3,4,1008153,0,1,6.65,0,True,...,False,True,False,False,False,False,True,False,True,False
1,24,306,7.4,1,1860642,0,0,6.82,0,False,...,True,False,False,False,False,False,False,True,False,False
2,33,409,7.1,1,1187409,0,1,8.22,0,True,...,False,False,False,False,False,True,False,True,False,False
3,31,408,7.5,1,1140053,1,1,9.18,1,False,...,True,False,False,False,False,True,False,False,True,False
4,28,13,6.8,4,1360395,0,1,7.11,0,False,...,False,True,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,34,126,7.2,1,2863917,1,1,9.23,1,True,...,False,False,True,False,False,False,False,False,False,False
96,24,118,6.1,1,1694682,0,1,7.63,0,True,...,False,False,False,True,False,False,False,False,True,False
97,34,48,8.3,3,2578007,1,1,7.68,1,True,...,False,False,False,False,True,False,False,False,False,True
98,28,89,7.8,1,1807506,0,0,9.21,0,True,...,False,True,False,False,False,False,True,False,True,False


In [ ]:
def area(rad):
    import math
    